# Kalman Class

In [3]:
import numpy as np

In [ ]:
class Kalman:
    def __init__(self, X, delta_t):
        self.X = X
        self.delta_t = delta_t

        self._F = np.identity(2)
        self._H = np.array([[1, 0]])
        self._Q = np.identity(2)
        self._P = np.identity(2)
        self._R = np.array([[1.0]])

    def _forward_step(self, z, t, X_prev, P_prev):
        # Set delta_t in the transition matrix (position += velocity * dt)
        self._F[0][1] = t

        # Project state forward using the motion model
        X_pred = self._F @ X_prev

        # Project covariance forward — uncertainty grows with time (Q scales with dt)
        P_pred = self._F @ P_prev @ self._F.T + self._Q * t

        # Compute Kalman gain — how much to trust the new observation vs the prediction
        S = self._H @ P_pred @ self._H.T + self._R
        K_gain = P_pred @ self._H.T @ np.linalg.inv(S)

        # Update state: prediction + gain * (what we saw - what we expected to see)
        X = X_pred + K_gain @ (z - self._H @ X_pred)

        # Update covariance — uncertainty shrinks after incorporating the observation
        P = (np.identity(2) - K_gain @ self._H) @ P_pred

        return X, X_pred, P, P_pred

    def _forward(self):
        var_r     = np.var(self.X, ddof=1)
        var_r_dot = np.var(np.diff(self.X), ddof=1)

        # Initialise state at zero — no prior belief on direction or momentum
        X_arr      = [np.array([[0], [0]])]
        X_pred_arr = [np.array([[0], [0]])]

        # Initialise uncertainty using sample variances of returns and their differences
        P0 = np.array([[var_r, 0], [0, var_r_dot]])
        P_arr      = [P0.copy()]
        P_pred_arr = [P0.copy()]

        for i in range(len(self.X)):
            X, X_pred, P, P_pred = self._forward_step(
                z      = self.X[i],
                t      = self.delta_t[i],
                X_prev = X_arr[i],
                P_prev = P_arr[i]
            )
            X_arr.append(X)
            X_pred_arr.append(X_pred)
            P_arr.append(P)
            P_pred_arr.append(P_pred)

        # Strip the initial conditions so every array aligns 1:1 with observations
        return X_arr[1:], X_pred_arr[1:], P_arr[1:], P_pred_arr[1:]

    def _smoother_step(self, X, X_pred_next, X_smooth_next,
                             P, P_pred_next, P_smooth_next, t):
        self._F[0][1] = t

        # Smoother gain — how much future information should revise the current estimate
        G_gain = P @ self._F.T @ np.linalg.inv(P_pred_next)

        # Revise state estimate using what the smoother believes about the next step
        smooth_X = X + G_gain @ (X_smooth_next - X_pred_next)

        # Revise covariance — future observations reduce uncertainty about the past
        smooth_P = P + G_gain @ (P_smooth_next - P_pred_next) @ G_gain.T

        return smooth_X, smooth_P, G_gain

    def _smoother(self):
        X_arr, X_pred_arr, P_arr, P_pred_arr = self._forward()

        # Initialise smoother at the last forward estimate — no future data beyond T
        smooth_X_arr   = [X_arr[-1]]
        smooth_P_arr   = [P_arr[-1]]
        smooth_cross_cov = []

        # Sweep backward from T-1 to 0
        for i in range(len(self.X) - 2, -1, -1):
            smooth_X, smooth_P, G_gain = self._smoother_step(
                X_arr[i],       X_pred_arr[i + 1],
                smooth_X_arr[-1],
                P_arr[i],       P_pred_arr[i + 1],
                smooth_P_arr[-1],
                self.delta_t[i]
            )

            # Cross covariance P_{t,t+1|T} — needed for Q update in EM
            # Must be computed before appending smooth_P so [-1] still refers to P_{t+1|T}
            smooth_cross_cov.append(G_gain @ smooth_P_arr[-1])
            smooth_X_arr.append(smooth_X)
            smooth_P_arr.append(smooth_P)

        return (
            smooth_X_arr[::-1],
            smooth_P_arr[::-1],
            smooth_cross_cov[::-1],
            X_pred_arr,
            P_pred_arr
        )

    def _log_likelihood(self):
        # Uses the forward pass only — innovations are one-step-ahead prediction errors
        _, X_pred_arr, _, P_pred_arr = self._forward()
        T  = len(self.X)
        ll = 0.0
        for i in range(T):
            # S is the total predicted observation variance (model uncertainty + sensor noise)
            S     = (self._H @ P_pred_arr[i] @ self._H.T + self._R)[0, 0]
            innov = self.X[i] - (self._H @ X_pred_arr[i])[0, 0]
            ll   += -0.5 * (np.log(S) + innov ** 2 / S)
        return ll

    def _EM_step(self):
        X_arr, P_arr, cross_cov, _, _ = self._smoother()
        T = len(self.X)

        # --- M-step: update R (observation noise) ---
        # R = average squared residual between smoothed state and raw observation,
        # corrected for the smoother's own remaining uncertainty (H @ P @ H.T term)
        R_new = np.zeros((1, 1))
        for i in range(T):
            residual       = self.X[i] - (self._H @ X_arr[i])[0, 0]
            R_new[0, 0]   += residual ** 2 + (self._H @ P_arr[i] @ self._H.T)[0, 0]
        self._R = R_new / T

        # --- M-step: update Q (process noise) ---
        # Q = average squared deviation between consecutive smoothed states,
        # corrected for their joint uncertainty via the cross covariance
        Q_new = np.zeros((2, 2))
        for i in range(1, T):
            xt      = X_arr[i]
            xt_prev = X_arr[i - 1]
            Pt      = P_arr[i]
            Pt_prev = P_arr[i - 1]

            # Ct combines the cross covariance and the outer product of adjacent states
            # cross_cov is length T-1, so index with i-1
            Ct = cross_cov[i - 1] + xt @ xt_prev.T

            Q_new += (
                Pt + xt @ xt.T
                - self._F @ Ct.T
                - Ct @ self._F.T
                + self._F @ (Pt_prev + xt_prev @ xt_prev.T) @ self._F.T
            )
        self._Q = Q_new / T

    def _EM(self, tol=1e-6, max_iter=100):
        ll_prev = self._log_likelihood()

        for i in range(max_iter):
            self._EM_step()
            ll = self._log_likelihood()

            if abs(ll - ll_prev) < tol:
                print(f"EM converged at iteration {i + 1}")
                break

            ll_prev = ll
        else:
            print(f"EM did not converge within {max_iter} iterations")

    def train(self):
        # Fit Q and R via EM on the full dataset
        self._EM()

        # Run the smoother one final time with the fitted parameters
        smooth_X, smooth_P, _, X_pred_arr, P_pred_arr = self._smoother()

        T = len(self.X)

        # --- Extract the five targets ---

        # Smoothed latent return: best estimate of true return on each day
        r_smooth = np.array([smooth_X[i][0, 0] for i in range(T)])

        # Smoothed latent momentum: best estimate of trend in returns on each day
        m_smooth = np.array([smooth_X[i][1, 0] for i in range(T)])

        # Posterior variance of return estimate: smoother's confidence in r_smooth
        # Small = confident label, Large = noisy label. Use as inverse sample weight: 1 / P_r
        P_r = np.array([smooth_P[i][0, 0] for i in range(T)])

        # Posterior variance of momentum estimate: smoother's confidence in m_smooth
        P_m = np.array([smooth_P[i][1, 1] for i in range(T)])

        # Normalised innovation: standardised surprise at each observation
        # = (observed return - predicted return) / sqrt(predicted observation variance)
        # Large magnitude = the market surprised the model. Fed as input to the HMM.
        innov = np.array([
            (self.X[i] - (self._H @ X_pred_arr[i])[0, 0]) /
            np.sqrt((self._H @ P_pred_arr[i] @ self._H.T + self._R)[0, 0])
            for i in range(T)
        ])

        return r_smooth, m_smooth, P_r, P_m, innov